# Notebook 06: Model Explainability

Explain model predictions using SHAP (tabular) and Grad-CAM (image), and perform error analysis.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import shap
import joblib
import tensorflow as tf
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from tensorflow.keras import Model

sns.set_style("whitegrid")
BASE_DIR = Path("..")
CLASSES = ["Normal", "Mild", "Moderate", "Severe"]
IMG_SIZE = (224, 224)
RANDOM_STATE = 42


## 2. Load Data and Preprocessors

In [ ]:
df = pd.read_csv(BASE_DIR / "data" / "metadata.csv")

le_label = LabelEncoder(); le_label.fit(CLASSES)
df["label"] = le_label.transform(df["diagnosis"])

le_gender = LabelEncoder()
df["gender_enc"] = le_gender.fit_transform(df["gender"])

scaler = StandardScaler()
df["age_scaled"] = scaler.fit_transform(df[["age"]])

df_train, df_temp = train_test_split(df, test_size=0.2, stratify=df["diagnosis"], random_state=RANDOM_STATE)
df_val, df_test   = train_test_split(df_temp, test_size=0.5, stratify=df_temp["diagnosis"], random_state=RANDOM_STATE)

X_train = df_train[["age", "gender_enc"]].values
X_test  = df_test[["age",  "gender_enc"]].values
y_train = df_train["label"].values
y_test  = df_test["label"].values


## 3. SHAP: Tabular Explainability

In [ ]:
# Load saved Random Forest model
rf_model = joblib.load("../models/saved_models/rf_model.pkl")

# Use the full tabular feature matrix
X_all = df[["age", "gender_enc"]].values
feature_names = ["Age (months)", "Gender (encoded)"]

# SHAP Tree Explainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# For multi-class, shap_values is a list of arrays (one per class)
print(f"SHAP values type: {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"Number of classes: {len(shap_values)}")
    print(f"Shape per class  : {shap_values[0].shape}")


In [ ]:
# SHAP Summary Plot — feature importance across all classes
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_names,
    class_names=CLASSES,
    plot_type="bar",
    show=False
)
plt.title("SHAP Feature Importance (Random Forest — All Classes)")
plt.tight_layout()
plt.savefig("../models/saved_models/shap_summary_bar.png", bbox_inches="tight")
plt.show()


In [ ]:
# SHAP beeswarm for Severe class (index 3)
severe_idx = le_label.transform(["Severe"])[0]
shap.summary_plot(
    shap_values[severe_idx], X_test,
    feature_names=feature_names,
    show=False
)
plt.title("SHAP Beeswarm — Severe Anemia Class")
plt.tight_layout()
plt.savefig("../models/saved_models/shap_beeswarm_severe.png", bbox_inches="tight")
plt.show()


In [ ]:
# Feature importance bar chart from RF
importances = rf_model.feature_importances_
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.barh(feature_names, importances, color=["#3498db", "#e74c3c"])
ax.set_title("Random Forest Feature Importance", fontweight="bold")
ax.set_xlabel("Importance")
for bar, val in zip(bars, importances):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center")
plt.tight_layout()
plt.savefig("../models/saved_models/rf_feature_importance.png", bbox_inches="tight")
plt.show()


## 4. Grad-CAM: Visual CNN Explainability

In [ ]:
def make_gradcam_heatmap(
    img_array: np.ndarray,
    model: Model,
    last_conv_layer_name: str = "Conv_1",
    pred_index: int = None,
) -> np.ndarray:
    """Compute Grad-CAM heatmap for a single image.

    Args:
        img_array: Preprocessed image of shape (1, H, W, 3).
        model: Keras model with a convolutional backbone.
        last_conv_layer_name: Name of the last convolutional layer to hook.
        pred_index: Class index to explain. If None, uses the argmax prediction.

    Returns:
        Normalised heatmap of shape (H, W) with values in [0, 1].
    """
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(original_img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    """Overlay a Grad-CAM heatmap on the original image."""
    heatmap_resized = np.array(
        Image.fromarray(np.uint8(255 * heatmap)).resize(
            (original_img.shape[1], original_img.shape[0])
        )
    )
    colormap = cm.get_cmap("jet")
    heatmap_rgb = colormap(heatmap_resized / 255.0)[:, :, :3]
    overlaid = (1 - alpha) * original_img / 255.0 + alpha * heatmap_rgb
    return np.clip(overlaid, 0.0, 1.0)


In [ ]:
# Load the fine-tuned visual model (if saved)
visual_model_path = "../models/saved_models/visual_model_finetuned.h5"
if not os.path.exists(visual_model_path):
    visual_model_path = "../../Notebook/models/mobilenetv2_finetuned_visual_model.h5"

visual_model = load_model(visual_model_path)
print(f"Loaded model from: {visual_model_path}")

# Find last conv layer name
conv_layer_name = None
for layer in reversed(visual_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        conv_layer_name = layer.name
        break
if conv_layer_name is None:
    # For MobileNetV2 via nested model
    conv_layer_name = "Conv_1"
print(f"Using last conv layer: {conv_layer_name}")


In [ ]:
def visualise_gradcam_grid(df_subset, n_samples=4):
    """Visualise Grad-CAM overlays for sample images from each severity class."""
    fig, axes = plt.subplots(len(CLASSES), n_samples, figsize=(n_samples * 4, len(CLASSES) * 3))
    colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]

    for row_idx, cls in enumerate(CLASSES):
        samples = df_subset[df_subset["diagnosis"] == cls].sample(
            min(n_samples, len(df_subset[df_subset["diagnosis"] == cls])),
            random_state=RANDOM_STATE
        )
        for col_idx in range(n_samples):
            ax = axes[row_idx, col_idx]
            if col_idx >= len(samples):
                ax.axis("off")
                continue
            row = samples.iloc[col_idx]
            img_path = str(BASE_DIR / row["image_path"])
            orig_img = np.array(Image.open(img_path).convert("RGB").resize((224, 224)))
            img_tensor = np.expand_dims(orig_img.astype(np.float32) / 255.0, axis=0)

            try:
                heatmap = make_gradcam_heatmap(img_tensor, visual_model, conv_layer_name)
                overlay = overlay_gradcam(orig_img, heatmap)
                ax.imshow(overlay)
            except Exception as e:
                ax.imshow(orig_img)
                ax.text(0.5, 0.05, f"CAM error: {e}", transform=ax.transAxes,
                        fontsize=7, ha="center", color="red")

            label_idx = row["label"]
            pred_idx = int(np.argmax(visual_model.predict(img_tensor, verbose=0)))
            pred_cls  = CLASSES[pred_idx]
            title_color = "green" if pred_cls == cls else "red"
            ax.set_title(f"True: {cls}\nPred: {pred_cls}",
                         color=title_color, fontsize=8, fontweight="bold")
            ax.axis("off")

    plt.suptitle("Grad-CAM Attention Maps — Conjunctiva Images",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../models/saved_models/gradcam_grid.png", bbox_inches="tight")
    plt.show()

visualise_gradcam_grid(df_test)


## 5. Error Analysis

In [ ]:
# Collect predictions on test set
y_preds, y_trues = [], []
for _, row in df_test.iterrows():
    img_path = str(BASE_DIR / row["image_path"])
    orig_img = np.array(Image.open(img_path).convert("RGB").resize((224, 224)))
    img_tensor = np.expand_dims(orig_img.astype(np.float32) / 255.0, axis=0)
    pred = int(np.argmax(visual_model.predict(img_tensor, verbose=0)))
    y_preds.append(pred)
    y_trues.append(row["label"])

y_trues = np.array(y_trues)
y_preds = np.array(y_preds)
misclassified_mask = y_trues != y_preds
misclassified_df = df_test.iloc[np.where(misclassified_mask)[0]].copy()
misclassified_df["predicted"] = [CLASSES[p] for p in y_preds[misclassified_mask]]
print(f"Total misclassified: {misclassified_mask.sum()} / {len(y_trues)}")
print("\nMisclassification by true class:")
print(misclassified_df["diagnosis"].value_counts())


In [ ]:
# Display misclassified images grouped by true class
n_show = 3
fig, axes = plt.subplots(len(CLASSES), n_show, figsize=(n_show * 4, len(CLASSES) * 3))
colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]

for row_idx, cls in enumerate(CLASSES):
    cls_errors = misclassified_df[misclassified_df["diagnosis"] == cls].head(n_show)
    for col_idx in range(n_show):
        ax = axes[row_idx, col_idx]
        if col_idx >= len(cls_errors):
            ax.axis("off"); continue
        row = cls_errors.iloc[col_idx]
        img = np.array(Image.open(str(BASE_DIR / row["image_path"])).convert("RGB").resize((224, 224)))
        ax.imshow(img)
        ax.set_title(f"True: {cls}\nPred: {row['predicted']}",
                     fontsize=8, color="red", fontweight="bold")
        ax.axis("off")

plt.suptitle("Misclassified Images — Error Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../models/saved_models/error_analysis_grid.png", bbox_inches="tight")
plt.show()


In [ ]:
# Summarise error patterns per class
print("=== Error Analysis Summary ===")
for cls in CLASSES:
    cls_errors = misclassified_df[misclassified_df["diagnosis"] == cls]
    total_cls = (df_test["diagnosis"] == cls).sum()
    err_rate = len(cls_errors) / total_cls if total_cls > 0 else 0
    print(f"\n{cls}: {len(cls_errors)}/{total_cls} errors ({err_rate:.1%})")
    if len(cls_errors) > 0:
        print("  Confused with:", cls_errors["predicted"].value_counts().to_dict())
